# DiffuSuite Colab Run Notebook

This notebook assumes your Google Drive contains the dataset at:

`/content/drive/MyDrive/diffu_suite/data/cifar10_dataset`

It will clone/update the code from GitHub, link Drive data into the Colab workspace, train/resume the cosine and linear DDPM checkpoints, export images/videos, and optionally launch the Gradio app.

## 1. Select GPU Runtime

Before running cells: `Runtime -> Change runtime type -> GPU`.

In [ ]:
!nvidia-smi

## 2. Mount Drive, Clone Repo, Link Dataset

If the GitHub repo is still private, paste a GitHub token when prompted. A fine-grained token with read access to `ColdVI/diffu_suite` is enough for cloning.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import getpass
import os
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/ColdVI/diffu_suite.git'
DRIVE_ROOT = Path('/content/drive/MyDrive/diffu_suite')
WORKSPACE = Path('/content/diffu_suite')
DATA_ROOT = DRIVE_ROOT / 'data' / 'cifar10_dataset'

if not DATA_ROOT.exists():
    raise FileNotFoundError(f'Dataset not found: {DATA_ROOT}')

if WORKSPACE.exists() and (WORKSPACE / '.git').exists():
    subprocess.run(['git', '-C', str(WORKSPACE), 'pull', '--ff-only'], check=True)
elif not WORKSPACE.exists():
    token = getpass.getpass('GitHub token for private repo, or press Enter if repo is public: ')
    clone_url = REPO_URL
    if token.strip():
        clone_url = REPO_URL.replace('https://', f'https://{token.strip()}@')
    subprocess.run(['git', 'clone', '--depth', '1', clone_url, str(WORKSPACE)], check=True)
else:
    raise RuntimeError(f'{WORKSPACE} exists but is not a Git checkout. Remove it or pick another workspace.')

os.chdir(WORKSPACE)

def replace_with_symlink(target: Path, source: Path) -> None:
    if target.is_symlink() or target.is_file():
        target.unlink()
    elif target.exists():
        shutil.rmtree(target)
    target.symlink_to(source, target_is_directory=True)

replace_with_symlink(WORKSPACE / 'data', DRIVE_ROOT / 'data')
(DRIVE_ROOT / 'runs').mkdir(parents=True, exist_ok=True)
replace_with_symlink(WORKSPACE / 'runs', DRIVE_ROOT / 'runs')

os.environ['DATA_ROOT'] = str(DATA_ROOT)
os.environ['DRIVE_ROOT'] = str(DRIVE_ROOT)
print('Workspace:', WORKSPACE)
print('Drive root:', DRIVE_ROOT)
print('Data root:', DATA_ROOT)
!git status --short --branch

## 3. Install Dependencies

In [ ]:
!python3 -m pip install -q -r requirements.txt

## 4. Validate Dataset

In [ ]:
!python3 scripts/validate_dataset.py --data-root "$DATA_ROOT" --hashes

## 5. Quick Setup Smoke Test

This tiny one-step run confirms that CUDA, the dataset, checkpointing, and imports are working before the long training job starts.

In [ ]:
!python3 training/train_ddpm.py \
  --data-root "$DATA_ROOT" \
  --device cuda \
  --output-dir "$DRIVE_ROOT/smoke_runs/setup_check" \
  --schedule cosine \
  --timesteps 4 \
  --epochs 1 \
  --batch-size 2 \
  --limit 8 \
  --workers 0 \
  --base-channels 8 \
  --conditioning-dim 32 \
  --channel-multipliers 1,2 \
  --num-res-blocks 1 \
  --attention-levels 1 \
  --attention-heads 1 \
  --dropout 0 \
  --max-steps 1 \
  --log-every 1 \
  --save-every 0 \
  --sample-every 0

## 6. Train or Resume Custom DDPM Checkpoints

Default settings train both cosine and linear schedules. If Colab time is limited, run cosine first, then set `TRAIN_COSINE = False` and `TRAIN_LINEAR = True` in a later session.

Memory guidance:
- Start with `BATCH_SIZE = 64`.
- Use `32` after CUDA out-of-memory.
- Try `128` on a stronger GPU.

In [ ]:
import shlex
import subprocess
from pathlib import Path

TRAIN_COSINE = True
TRAIN_LINEAR = True

EPOCHS = 100
BATCH_SIZE = 64
WORKERS = 2
SAVE_EVERY = 1000
SAMPLE_EVERY = 2000

def run_command(cmd):
    print(' '.join(shlex.quote(str(part)) for part in cmd))
    subprocess.run([str(part) for part in cmd], check=True)

def train_schedule(schedule: str):
    output_dir = DRIVE_ROOT / 'runs' / f'cifar10_{schedule}'
    latest = output_dir / 'checkpoints' / 'latest.pt'
    cmd = [
        'python3', 'training/train_ddpm.py',
        '--data-root', DATA_ROOT,
        '--schedule', schedule,
        '--output-dir', output_dir,
        '--batch-size', BATCH_SIZE,
        '--epochs', EPOCHS,
        '--workers', WORKERS,
        '--save-every', SAVE_EVERY,
        '--sample-every', SAMPLE_EVERY,
        '--device', 'cuda',
    ]
    if latest.exists():
        cmd.extend(['--resume', latest])
        print(f'Resuming {schedule} from {latest}')
    run_command(cmd)

if TRAIN_COSINE:
    train_schedule('cosine')
if TRAIN_LINEAR:
    train_schedule('linear')

## 7. Export Generated Samples and Reverse Videos

In [ ]:
from pathlib import Path
import subprocess

(DRIVE_ROOT / 'artifacts' / 'generated').mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'artifacts' / 'videos').mkdir(parents=True, exist_ok=True)

def export_schedule(schedule: str):
    checkpoint = DRIVE_ROOT / 'runs' / f'cifar10_{schedule}' / 'checkpoints' / 'latest.pt'
    if not checkpoint.exists():
        print(f'Skipping {schedule}: missing checkpoint {checkpoint}')
        return
    cmd = [
        'python3', 'inference/sample_custom.py', checkpoint,
        '--device', 'cuda',
        '--output', DRIVE_ROOT / 'artifacts' / 'generated' / f'cifar10_{schedule}.png',
        '--trajectory-stem', DRIVE_ROOT / 'artifacts' / 'videos' / f'cifar10_{schedule}_reverse',
        '--guidance-scale', '1.0',
    ]
    subprocess.run([str(part) for part in cmd], check=True)

export_schedule('cosine')
export_schedule('linear')

## 8. Export Diagnostics and Restoration Examples

In [ ]:
!python3 utils/generate_readme_assets.py \
  --data-root "$DATA_ROOT" \
  --output-dir "$DRIVE_ROOT/artifacts/examples"

In [ ]:
from PIL import Image, ImageDraw
import subprocess

checkpoint = DRIVE_ROOT / 'runs' / 'cifar10_cosine' / 'checkpoints' / 'latest.pt'
if not checkpoint.exists():
    print(f'Skipping restoration: missing {checkpoint}')
else:
    restored_dir = DRIVE_ROOT / 'artifacts' / 'restored'
    video_dir = DRIVE_ROOT / 'artifacts' / 'videos'
    restored_dir.mkdir(parents=True, exist_ok=True)
    video_dir.mkdir(parents=True, exist_ok=True)

    source = WORKSPACE / 'artifacts' / 'examples' / 'source_ship.png'
    mask = restored_dir / 'center_mask.png'
    mask_image = Image.new('L', (32, 32), 0)
    ImageDraw.Draw(mask_image).rectangle((8, 8, 23, 23), fill=255)
    mask_image.save(mask)

    subprocess.run([
        'python3', 'inference/restore_custom.py', checkpoint, source,
        '--device', 'cuda',
        '--task', 'inpaint',
        '--mask', mask,
        '--class-id', '8',
        '--output', restored_dir / 'inpainted_ship.png',
        '--trajectory-stem', video_dir / 'inpainting_ship',
    ], check=True)

    subprocess.run([
        'python3', 'inference/restore_custom.py', checkpoint, source,
        '--device', 'cuda',
        '--task', 'super-res',
        '--class-id', '8',
        '--downsample-factor', '4',
        '--output', restored_dir / 'super_res_ship.png',
    ], check=True)

## 9. Display Key Outputs

In [ ]:
from IPython.display import Image as IPImage, Video, display

image_paths = [
    DRIVE_ROOT / 'artifacts' / 'generated' / 'cifar10_cosine.png',
    DRIVE_ROOT / 'artifacts' / 'generated' / 'cifar10_linear.png',
    DRIVE_ROOT / 'artifacts' / 'examples' / 'forward_degradation.png',
    DRIVE_ROOT / 'artifacts' / 'restored' / 'inpainted_ship.png',
    DRIVE_ROOT / 'artifacts' / 'restored' / 'super_res_ship.png',
]
for path in image_paths:
    if path.exists():
        print(path)
        display(IPImage(filename=str(path)))

video_path = DRIVE_ROOT / 'artifacts' / 'videos' / 'cifar10_cosine_reverse.mp4'
if video_path.exists():
    display(Video(str(video_path), embed=True))

## 10. Launch Gradio App

The app default checkpoint path `runs/cifar10_cosine/checkpoints/latest.pt` works because this notebook symlinked `runs/` to Drive.

In [ ]:
!python3 app.py --host 0.0.0.0 --share

## 11. Optional Advanced Diffusers Setup

Run this only if you want the Production Studio tab: LoRA inference/training and Canny ControlNet.

In [ ]:
from pathlib import Path
import subprocess
import sys

RUN_ADVANCED_SETUP = False

if RUN_ADVANCED_SETUP:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-advanced.txt'], check=True)
    if not Path('third_party/diffusers').exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/huggingface/diffusers', 'third_party/diffusers'], check=True)
    print('Advanced setup complete. Relaunch app.py when ready.')

## 12. Optional LoRA Dry Run

Place 10-20 concept images under:

`/content/drive/MyDrive/diffu_suite/data/lora/my_concept`

Then set `RUN_LORA_DRY_RUN = True`. Remove `--dry-run` manually when the printed command looks correct.

In [ ]:
import subprocess

RUN_LORA_DRY_RUN = False
LORA_CONCEPT_DIR = DRIVE_ROOT / 'data' / 'lora' / 'my_concept'

if RUN_LORA_DRY_RUN:
    subprocess.run([
        'python3', 'advanced/train_lora.py', str(LORA_CONCEPT_DIR),
        '--instance-prompt', 'a photo of sks concept',
        '--output-dir', str(DRIVE_ROOT / 'runs' / 'lora' / 'my_concept'),
        '--dry-run',
    ], check=True)